# Generate Synthetic Data from Cleveland Heart Disease (303 rows)

**Goal**: fit CTGAN on the 303-row UCI Cleveland dataset and generate a large
synthetic training set — then compare CatBoost trained on *our* synthetic data
vs CatBoost trained on the *competition's* synthetic data (630k rows).

**Run on Kaggle** (CTGAN + internet access available). Download the output CSVs.

Generates:
- `cleveland_synthetic_100k.csv` — quick sanity check / fast local test
- `cleveland_synthetic_630k.csv` — full comparison matching competition size

## Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import io
import urllib.request
from pathlib import Path

# CTGAN — works with both standalone ctgan and sdv v0.x / v1.x
try:
    from ctgan import CTGAN
    print('Using: ctgan standalone')
except ImportError:
    from sdv.tabular import CTGAN
    print('Using: sdv.tabular.CTGAN')

OUTPUT_DIR = Path('/kaggle/working')
print('Output dir:', OUTPUT_DIR)

## Load Cleveland Data

303 rows from the UCI Heart Disease repository. Uses the same integer codes
as the competition data (sex 0/1, chest_pain_type 1-4, thallium 3/6/7, etc.).

In [ ]:
COLS = ['age', 'sex', 'chest_pain_type', 'bp', 'cholesterol', 'fbs_over_120',
        'ekg_results', 'max_hr', 'exercise_angina', 'st_depression',
        'slope_of_st', 'number_of_vessels_fluro', 'thallium', 'heart_disease']

CAT_COLS = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
            'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro',
            'thallium', 'heart_disease']

UCI_URL = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'

try:
    raw = urllib.request.urlopen(UCI_URL).read().decode()
    cleve = pd.read_csv(io.StringIO(raw), header=None, names=COLS, na_values='?')
    print('Loaded from UCI URL')
except Exception as e:
    print(f'URL fetch failed ({e}), trying local path...')
    cleve = pd.read_csv('/kaggle/input/uci-cleveland/processed.cleveland.data',
                        header=None, names=COLS, na_values='?')

# Drop 6 rows with missing values
cleve = cleve.dropna().reset_index(drop=True)

# Binarize target: 0=absent, 1-4=present
cleve['heart_disease'] = (cleve['heart_disease'] > 0).astype(int)

# Cast categoricals to int (CTGAN needs to know which columns are discrete)
for col in CAT_COLS:
    cleve[col] = cleve[col].astype(int)

print(f'Cleveland: {cleve.shape}')
print(f'Positive rate: {cleve.heart_disease.mean():.3f}')
print(f'\nValue counts per categorical:')
for col in CAT_COLS:
    print(f'  {col}: {sorted(cleve[col].unique().tolist())}')

## Quick EDA — Distributions to Validate Against Later

In [ ]:
NUM_COLS = [c for c in COLS if c not in CAT_COLS]

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, NUM_COLS):
    cleve[col].hist(bins=20, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(col)
plt.suptitle('Cleveland (303 rows) — Numeric Feature Distributions')
plt.tight_layout()
plt.show()

print('\nBasic stats:')
print(cleve[NUM_COLS].describe().round(2))

## Fit CTGAN

CTGAN (Conditional Tabular GAN) is the likely family of generator the competition used.
Training on 303 rows — small dataset, so we use moderate epochs to avoid mode collapse.

In [ ]:
model = CTGAN(
    epochs=500,
    batch_size=100,            # small — dataset is only 303 rows
    generator_dim=(256, 256),
    discriminator_dim=(256, 256),
    generator_lr=2e-4,
    discriminator_lr=2e-4,
    discriminator_steps=1,
    log_frequency=True,
    verbose=True,
)

model.fit(cleve, discrete_columns=CAT_COLS)
print('\nCTGAN training complete.')

## Generate Synthetic Data

In [ ]:
print('Generating 100k rows...')
synth_100k = model.sample(100_000)
print(f'  shape: {synth_100k.shape}')

print('Generating 630k rows (competition size)...')
synth_630k = model.sample(630_000)
print(f'  shape: {synth_630k.shape}')

## Validate — Compare Distributions

Check that the synthetic data's marginal distributions resemble Cleveland.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.flat, NUM_COLS):
    cleve[col].hist(bins=20, ax=ax, alpha=0.5, label='Cleveland (real)', density=True)
    synth_100k[col].hist(bins=20, ax=ax, alpha=0.5, label='Synthetic 100k', density=True)
    ax.set_title(col)
    ax.legend(fontsize=7)
plt.suptitle('Cleveland (real) vs CTGAN Synthetic — Numeric Distributions')
plt.tight_layout()
plt.show()

# Positive rate check
print(f'Positive rate — Cleveland: {cleve.heart_disease.mean():.3f}')
print(f'Positive rate — Synth 100k: {synth_100k.heart_disease.mean():.3f}')
print(f'Positive rate — Synth 630k: {synth_630k.heart_disease.mean():.3f}')

# Categorical value coverage
print('\nCategorical coverage (synthetic vs real):')
for col in CAT_COLS:
    real_vals  = set(cleve[col].unique())
    synth_vals = set(synth_100k[col].unique())
    print(f'  {col}: real={sorted(real_vals)}  synth={sorted(synth_vals)}')

## Post-process & Save

Clip numeric features to realistic ranges, cast categoricals to int.

In [ ]:
RANGES = {
    'age':           (20, 100),
    'bp':            (80, 250),
    'cholesterol':   (100, 600),
    'max_hr':        (60, 220),
    'st_depression': (0.0, 10.0),
}

def postprocess(df):
    df = df.copy()
    for col, (lo, hi) in RANGES.items():
        df[col] = df[col].clip(lo, hi)
    for col in CAT_COLS:
        df[col] = df[col].round().astype(int)
    # Ensure heart_disease is 0/1
    df['heart_disease'] = df['heart_disease'].clip(0, 1)
    return df[COLS]  # enforce column order

synth_100k = postprocess(synth_100k)
synth_630k = postprocess(synth_630k)

# Save
path_100k = OUTPUT_DIR / 'cleveland_synthetic_100k.csv'
path_630k = OUTPUT_DIR / 'cleveland_synthetic_630k.csv'

synth_100k.to_csv(path_100k, index=False)
synth_630k.to_csv(path_630k, index=False)

print(f'Saved: {path_100k}  ({path_100k.stat().st_size / 1e6:.1f} MB)')
print(f'Saved: {path_630k}  ({path_630k.stat().st_size / 1e6:.1f} MB)')
print('\nDownload from Kaggle output tab.')

## Preview

In [ ]:
print('=== Cleveland (real, first 5 rows) ===')
print(cleve.head())
print('\n=== Synthetic 100k (first 5 rows) ===')
print(synth_100k.head())
print('\n=== Synthetic stats comparison ===')
comparison = pd.concat([
    cleve[NUM_COLS].describe().round(2).add_suffix(' [real]'),
    synth_100k[NUM_COLS].describe().round(2).add_suffix(' [synth]'),
], axis=1)
print(comparison[sorted(comparison.columns)])